<a href="https://colab.research.google.com/github/FReddy116/ai-powered-resume-matcher/blob/main/Final_Ai_resume_matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install  pdfplumber faiss-cpu sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.4 MB/s eta 0:00:00


In [2]:
import os
folder_path = "/content"
for file in os.listdir(folder_path):
  if file.endswith(".pdf"):
    os.remove(os.path.join(folder_path, file))
print("Old files removed")

Old files removed


In [3]:
from google.colab import files
uploaded = files.upload()

Saving cloud_engineer_resume.pdf to cloud_engineer_resume.pdf
Saving resume_backend_java.pdf to resume_backend_java.pdf
Saving resume_cybersecurity.pdf to resume_cybersecurity.pdf
Saving resume_data_scientist.pdf to resume_data_scientist.pdf
Saving resume_devops.pdf to resume_devops.pdf
Saving resume_frontend.pdf to resume_frontend.pdf
Saving software_developer_resume.pdf to software_developer_resume.pdf


In [4]:
import pdfplumber

def extract_text_from_pdf(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    return text

In [5]:
import re

def extract_sections(text):
    sections = {
        "skills": "",
        "experience": "",
        "projects": "",
        "certifications": ""
    }

    text_lower = text.lower()


    patterns = {
        "skills": ["skills", "technical skills", "key skills"],
        "experience": ["experience", "work experience", "professional experience"],
        "projects": ["projects", "personal projects"],
        "certifications": ["certifications", "certificates"]
    }

    for key, keywords in patterns.items():
        for kw in keywords:
            pattern = rf"{kw}(.+?)(\n[A-Z ]{{3,}}|\Z)"
            match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            if match:
                sections[key] = match.group(1).strip()
                break

    return sections

In [6]:
resume_texts = {}

for file_name in uploaded.keys():
    text = extract_text_from_pdf(file_name)
    resume_texts[file_name] = text

In [7]:
def clean_text(text):
    text = text.lower()
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text[:2000]

In [8]:
structured_data = {}

for file_name, text in resume_texts.items():
    structured_data[file_name] = extract_sections(text)

In [9]:
cleaned_docs = []

for file_name, sections in structured_data.items():
    combined_text = clean_text(" ".join(sections.values()))

    cleaned_docs.append({
        "file": file_name,
        "text": combined_text
    })

In [10]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device ='cpu')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
import numpy as np

documents = []
embeddings = []

texts = [doc["text"] for doc in cleaned_docs]
documents = [doc["file"] for doc in cleaned_docs]

embeddings = embed_model.encode(
    texts,
    batch_size=4,
    show_progress_bar=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
import numpy as np
import faiss

embeddings = np.array(embeddings).astype('float32')

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [14]:
query = input("Enter Job Description: ")

query_embedding = embed_model.encode([query]).astype('float32')

D, I = index.search(query_embedding, k=min(5, len(documents)))

print("\n Top Matching resumes:\n")

for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):

    score = 1 / (1 + dist)
    percentage = round(score * 100, 2)

    print(f"Rank {rank}: {documents[idx]}")
    pc=round(percentage,2)
    print(f"Match Score: {pc}%")
    print("-" * 40)

Enter Job Description: We are looking for a Backend Engineer with experience in microservices architecture. Required skills include Java, Spring Boot, REST APIs, and experience building scalable backend services. Knowledge of cloud platforms like AWS and container technologies such as Docker is a plus.

 Top Matching resumes:

Rank 1: resume_devops.pdf
Match Score: 52.13999938964844%
----------------------------------------
Rank 2: resume_backend_java.pdf
Match Score: 51.70000076293945%
----------------------------------------
Rank 3: software_developer_resume.pdf
Match Score: 40.599998474121094%
----------------------------------------
Rank 4: resume_frontend.pdf
Match Score: 39.529998779296875%
----------------------------------------
Rank 5: resume_data_scientist.pdf
Match Score: 39.4900016784668%
----------------------------------------
